# Analysing LLM-as-a-Judge Evaluations of the Generated Dialogues

For the evaluation of every model of the three judging models the same pipeline is applied:
- extracting valid evaluations
- extract applied labels
- calculate accuracy and F1

Then, combine the analyses, calculate consesus accuracy and F1, calculate Krippendorffs $\alpha$ and McNemar's Test for statistical significance.

In [ ]:
import pandas as pd
import json
import re

def extract_valid_dialogues(data):
    """
    Searches JSON structures for dialogues
    Repairs broken Strings with Regex
    """
    valid_dialogues = []
    
    if isinstance(data, list):
        for item in data:
            # If raw string instead of json -> Regex
            if isinstance(item, str):
                try:
                    match = re.search(r'\{.*\}', item, re.DOTALL)
                    if match:
                        item = json.loads(match.group(0))
                except Exception:
                    continue # ignores if repair doesn't work
            
            
            if isinstance(item, (dict, list)):
                valid_dialogues.extend(extract_valid_dialogues(item))
                
    elif isinstance(data, dict):
        
        if "global_id" in data:
            valid_dialogues.append(data)
        else:
            
            for value in data.values():
                valid_dialogues.extend(extract_valid_dialogues(value))
                
    return valid_dialogues

# ==========================================
# 1. load GROUND TRUTH 
# ==========================================
# ground truth labels are identical over all dialogue files
with open("C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/json_ground_truth_dias/dialogues_ministral_base.json", "r", encoding="utf-8") as f:
    raw_inputs = json.load(f)

# initialise master table
df_master_qwen3_32B = pd.DataFrame([{
    "global_id": i + 1,
    "Szenario": d["Szenario"],
    "Run": d["Run"],
    "true_MBTI_A": d["MBTI_A"],
    "true_MBTI_B": d["MBTI_B"]
} for i, d in enumerate(raw_inputs)])

# ==========================================
# 2. load evals
# ==========================================
eval_files = {
    "qwen_base": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3-32/judgments_qwen3-32b_dialogues_qwen_base.json",
    "qwen_lora1": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3-32/judgments_qwen3-32b_dialogues_qwen_lora1_correct.json",
    "qwen_lora2": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3-32/judgments_qwen3-32b_dialogues_qwen_lora2.json",
    "mistral_base": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3-32/judgments_qwen3-32b_dialogues_ministral_base.json",
    "mistral_lora1": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3-32/judgments_qwen3-32b_dialogues_ministral_lora1.json",
    "mistral_lora2": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3-32/judgments_qwen3-32b_dialogues_ministral_lora2.json"
}

for variant_name, filepath in eval_files.items():
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Not able to load file: {filepath} -> {e}")
        continue
        
    rows = []
    
    # extract valid evals
    clean_dialogues = extract_valid_dialogues(data)
    
    for diag in clean_dialogues:
        global_id = diag.get("global_id")
        conclusion = diag.get("final_conclusion", {})
        
        pred_a = conclusion.get("person_A", {}).get("type (best guess)")
        pred_b = conclusion.get("person_B", {}).get("type (best guess)")
        
        rows.append({
            "global_id": global_id,
            f"pred_A_{variant_name}": pred_a,
            f"pred_B_{variant_name}": pred_b
        })
        
    df_temp = pd.DataFrame(rows)
    # merge table with global_id
    df_master_qwen3_32B = pd.merge(df_master_qwen3_32B, df_temp, on="global_id", how="left")

display(df_master_qwen3_32B.head())

,global_id,Szenario,Run,true_MBTI_A,true_MBTI_B,pred_A_qwen_base,pred_B_qwen_base,pred_A_qwen_lora1,pred_B_qwen_lora1,pred_A_qwen_lora2,pred_B_qwen_lora2,pred_A_mistral_base,pred_B_mistral_base,pred_A_mistral_lora1,pred_B_mistral_lora1,pred_A_mistral_lora2,pred_B_mistral_lora2
0,1,Work Place - Low Urgency,1,INTJ,ENFP,ENTJ,ISFP,ENFP,ESFJ,ENTJ,INFP,ISTJ,ENFP,ENTP,INFP,ENTJ,ESFJ
1,2,Work Place - Low Urgency,2,INTJ,ENFP,ESTJ,ESFJ,NaN,NaN,ENTP,INFP,ENTJ,ENFP,ISFJ,ENFP,ENTP,INFJ
2,3,Work Place - Low Urgency,3,INTJ,ENFP,ESTJ,ENFP,ENTJ,ISFP,ESTJ,ESFJ,INTJ,ENFP,NaN,NaN,ESTJ,INFP
3,4,Work Place - Low Urgency,4,INTJ,ENFP,ISTJ,ENFP,ESTJ,INFP,INFJ,ESFP,NaN,NaN,ESTJ,INFP,ENTP,INFP
4,5,Work Place - Low Urgency,5,INTJ,ENFP,INTJ,ENFP,ENTJ,ESFP,ESTJ,INFP,ISTJ,ENFP,ENFJ,INFP,ENTP,ISFJ


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import re

# cleaning function
def clean_mbti(val):
    if pd.isna(val) or not isinstance(val, str):
        return None
    clean_val = val.strip().upper()
    match = re.search(r'[IE][SN][TF][JP]', clean_val)
    if match:
        return match.group(0)
    return None

variant_names = [
    "mistral_base", "mistral_lora1", "mistral_lora2", 
    "qwen_base", "qwen_lora1", "qwen_lora2"
]

dim_names = ['E/I', 'S/N', 'T/F', 'J/P']
results = []

for var in variant_names:
    true_labels = df_master_qwen3_32B['true_MBTI_A'].tolist() + df_master_qwen3_32B['true_MBTI_B'].tolist()
    pred_col_A = f"pred_A_{var}"
    pred_col_B = f"pred_B_{var}"
    
    if pred_col_A not in df_master_qwen3_32B.columns or pred_col_B not in df_master_qwen3_32B.columns:
        continue
        
    pred_labels = df_master_qwen3_32B[pred_col_A].apply(clean_mbti).tolist() + \
                  df_master_qwen3_32B[pred_col_B].apply(clean_mbti).tolist()
    
    # filter valid pairs
    valid_true = []
    valid_pred = []
    for t, p in zip(true_labels, pred_labels):
        if p is not None and t is not None and len(t) == 4 and len(p) == 4:
            valid_true.append(t)
            valid_pred.append(p)
            
    if len(valid_true) == 0:
        continue
        
    # base data for this row
    row_data = {
        "Model Variant": var.replace("_", " ").title(),
        "N": len(valid_true)
    }
    
    # evals per dimension
    for i, dim in enumerate(dim_names):
    
        t_dim = [t[i] for t in valid_true]
        p_dim = [p[i] for p in valid_pred]
        
        acc = accuracy_score(t_dim, p_dim)
        f1 = f1_score(t_dim, p_dim, average='macro')
        
        row_data[f"{dim} Acc"] = round(acc, 3)
        row_data[f"{dim} F1"] = round(f1, 3)
        
    results.append(row_data)


df_results_qwen3_32B = pd.DataFrame(results)
display(df_results_qwen3_32B)

,Model Variant,N,E/I Acc,E/I F1,S/N Acc,S/N F1,T/F Acc,T/F F1,J/P Acc,J/P F1
0,Mistral Base,404,0.579,0.575,0.601,0.598,0.792,0.792,0.782,0.782
1,Mistral Lora1,432,0.421,0.421,0.546,0.543,0.704,0.704,0.657,0.657
2,Mistral Lora2,427,0.424,0.421,0.546,0.541,0.740,0.740,0.665,0.665
3,Qwen Base,425,0.499,0.489,0.638,0.638,0.774,0.774,0.755,0.752
4,Qwen Lora1,418,0.467,0.461,0.512,0.511,0.778,0.777,0.699,0.698
5,Qwen Lora2,418,0.455,0.450,0.548,0.548,0.727,0.727,0.694,0.693


In [ ]:
def extract_valid_dialogues(data):
    """
    Searches JSON structures for dialogues
    Repairs broken Strings with Regex
    """
    valid_dialogues = []
    
    if isinstance(data, list):
        for item in data:
            # If raw string instead of json -> Regex
            if isinstance(item, str):
                try:
                    match = re.search(r'\{.*\}', item, re.DOTALL)
                    if match:
                        item = json.loads(match.group(0))
                except Exception:
                    continue # ignores if repair doesn't work
            
            
            if isinstance(item, (dict, list)):
                valid_dialogues.extend(extract_valid_dialogues(item))
                
    elif isinstance(data, dict):
        
        if "global_id" in data:
            valid_dialogues.append(data)
        else:

            for value in data.values():
                valid_dialogues.extend(extract_valid_dialogues(value))
                
    return valid_dialogues

# ==========================================
# 1. load GROUND TRUTH 
# ==========================================
# ground truth labels are identical over all dialogue file
with open("C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/json_ground_truth_dias/dialogues_ministral_base.json", "r", encoding="utf-8") as f:
    raw_inputs = json.load(f)

# initialise master table
df_master_qwen3_30B = pd.DataFrame([{
    "global_id": i + 1,
    "Szenario": d["Szenario"],
    "Run": d["Run"],
    "true_MBTI_A": d["MBTI_A"],
    "true_MBTI_B": d["MBTI_B"]
} for i, d in enumerate(raw_inputs)])

# ==========================================
# 2. load evals
# ==========================================
eval_files = {
    "qwen_base": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3_30/judgments_qwen3-30b-a3b-instruct-2507_dialogues_qwen_base.json",
    "qwen_lora1": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3_30/judgments_qwen3-30b-a3b-instruct-2507_dialogues_qwen_lora1_correct.json",
    "qwen_lora2": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3_30/judgments_qwen3-30b-a3b-instruct-2507_dialogues_qwen_lora2.json",
    "mistral_base": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3_30/judgments_qwen3-30b-a3b-instruct-2507_dialogues_ministral_base.json",
    "mistral_lora1": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3_30/judgments_qwen3-30b-a3b-instruct-2507_dialogues_ministral_lora1.json",
    "mistral_lora2": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/qwen3_30/judgments_qwen3-30b-a3b-instruct-2507_dialogues_ministral_lora2.json"
}


for variant_name, filepath in eval_files.items():
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Konnte Datei nicht laden: {filepath} -> {e}")
        continue
        
    rows = []
    
    # extract valid evals
    clean_dialogues = extract_valid_dialogues(data)
    
    for diag in clean_dialogues:
        global_id = diag.get("global_id")
        conclusion = diag.get("final_conclusion", {})
        
        pred_a = conclusion.get("person_A", {}).get("type (best guess)")
        pred_b = conclusion.get("person_B", {}).get("type (best guess)")
        
        rows.append({
            "global_id": global_id,
            f"pred_A_{variant_name}": pred_a,
            f"pred_B_{variant_name}": pred_b
        })
        
    df_temp = pd.DataFrame(rows)
    
    # merge table with global_id
    df_master_qwen3_30B = pd.merge(df_master_qwen3_30B, df_temp, on="global_id", how="left")

display(df_master_qwen3_30B.head())

,global_id,Szenario,Run,true_MBTI_A,true_MBTI_B,pred_A_qwen_base,pred_B_qwen_base,pred_A_qwen_lora1,pred_B_qwen_lora1,pred_A_qwen_lora2,pred_B_qwen_lora2,pred_A_mistral_base,pred_B_mistral_base,pred_A_mistral_lora1,pred_B_mistral_lora1,pred_A_mistral_lora2,pred_B_mistral_lora2
0,1,Work Place - Low Urgency,1,INTJ,ENFP,ENTJ,INFP,ENFP,ENFP,ENTJ,INFP,INTJ,ENFP,INTP,ENTJ,ENTP,ISTJ
1,2,Work Place - Low Urgency,2,INTJ,ENFP,ENTJ,INFJ,ENFP,INFJ,ENFJ,INFJ,ENTJ,ENFP,ENTJ,ENFP,ENTJ,INFP
2,3,Work Place - Low Urgency,3,INTJ,ENFP,ENTJ,ENFP,ENTJ,INFJ,ENTJ,INFJ,INTJ,ENFP,ENTJ,INFP,ENTP,INFP
3,4,Work Place - Low Urgency,4,INTJ,ENFP,ENTJ,ENFP,ENTJ,INFP,INFP,ENFP,ENTJ,ENFP,ENTJ,INFP,ENTJ,INFP
4,5,Work Place - Low Urgency,5,INTJ,ENFP,ENTJ,ENFP,ESTJ,ENFP,ENTJ,INFP,ENFP,ISTJ,INTJ,ENFP,ENTP,INFP


In [ ]:
# cleaning function
def clean_mbti(val):
    if pd.isna(val) or not isinstance(val, str):
        return None
    clean_val = val.strip().upper()
    match = re.search(r'[IE][SN][TF][JP]', clean_val)
    if match:
        return match.group(0)
    return None

variant_names = [
    "mistral_base", "mistral_lora1", "mistral_lora2", 
    "qwen_base", "qwen_lora1", "qwen_lora2"
]

dim_names = ['E/I', 'S/N', 'T/F', 'J/P']
results = []

for var in variant_names:
    true_labels = df_master_qwen3_30B['true_MBTI_A'].tolist() + df_master_qwen3_30B['true_MBTI_B'].tolist()
    pred_col_A = f"pred_A_{var}"
    pred_col_B = f"pred_B_{var}"
    
    if pred_col_A not in df_master_qwen3_30B.columns or pred_col_B not in df_master_qwen3_30B.columns:
        continue
        
    pred_labels = df_master_qwen3_30B[pred_col_A].apply(clean_mbti).tolist() + \
                  df_master_qwen3_30B[pred_col_B].apply(clean_mbti).tolist()
    
    # filter valid pairs
    valid_true = []
    valid_pred = []
    for t, p in zip(true_labels, pred_labels):
        if p is not None and t is not None and len(t) == 4 and len(p) == 4:
            valid_true.append(t)
            valid_pred.append(p)
            
    if len(valid_true) == 0:
        continue
        
    # base data for this row
    row_data = {
        "Model Variant": var.replace("_", " ").title(),
        "N": len(valid_true)
    }
    
    # evals per dimension
    for i, dim in enumerate(dim_names):
        
        t_dim = [t[i] for t in valid_true]
        p_dim = [p[i] for p in valid_pred]
        
        acc = accuracy_score(t_dim, p_dim)
        
        f1 = f1_score(t_dim, p_dim, average='macro')
        
        row_data[f"{dim} Acc"] = round(acc, 3)
        row_data[f"{dim} F1"] = round(f1, 3)
        
    results.append(row_data)


df_results_qwen3_30B = pd.DataFrame(results)
display(df_results_qwen3_32B)

,Model Variant,N,E/I Acc,E/I F1,S/N Acc,S/N F1,T/F Acc,T/F F1,J/P Acc,J/P F1
0,Mistral Base,404,0.579,0.575,0.601,0.598,0.792,0.792,0.782,0.782
1,Mistral Lora1,432,0.421,0.421,0.546,0.543,0.704,0.704,0.657,0.657
2,Mistral Lora2,427,0.424,0.421,0.546,0.541,0.740,0.740,0.665,0.665
3,Qwen Base,425,0.499,0.489,0.638,0.638,0.774,0.774,0.755,0.752
4,Qwen Lora1,418,0.467,0.461,0.512,0.511,0.778,0.777,0.699,0.698
5,Qwen Lora2,418,0.455,0.450,0.548,0.548,0.727,0.727,0.694,0.693


In [ ]:
def extract_valid_dialogues(data):
    """
    Searches JSON structures for dialogues
    Repairs broken Strings with Regex
    """
    valid_dialogues = []
    
    if isinstance(data, list):
        for item in data:
            # If raw string instead of json -> Regex
            if isinstance(item, str):
                try:
                    match = re.search(r'\{.*\}', item, re.DOTALL)
                    if match:
                        item = json.loads(match.group(0))
                except Exception:
                    continue # ignores if repair doesn't work

            
            if isinstance(item, (dict, list)):
                valid_dialogues.extend(extract_valid_dialogues(item))
                
    elif isinstance(data, dict):

        if "global_id" in data:
            valid_dialogues.append(data)
        else:

            for value in data.values():
                valid_dialogues.extend(extract_valid_dialogues(value))
                
    return valid_dialogues

# ==========================================
# 1. load GROUND TRUTH 
# ==========================================
# ground truth labels are identical over all dialogue file
with open("C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/json_ground_truth_dias/dialogues_ministral_base.json", "r", encoding="utf-8") as f:
    raw_inputs = json.load(f)

# initialise master table
df_master_mistral31_24B = pd.DataFrame([{
    "global_id": i + 1,
    "Szenario": d["Szenario"],
    "Run": d["Run"],
    "true_MBTI_A": d["MBTI_A"],
    "true_MBTI_B": d["MBTI_B"]
} for i, d in enumerate(raw_inputs)])

# ==========================================
# 2. load evals
# ==========================================
eval_files = {
    "qwen_base": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/mistral/judgments_mistral-small-3.1-24b-instruct-2503_dialogues_qwen_base.json",
    "qwen_lora1": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/mistral/judgments_mistral-small-3.1-24b-instruct-2503_dialogues_qwen_lora1_correct.json",
    "qwen_lora2": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/mistral/judgments_mistral-small-3.1-24b-instruct-2503_dialogues_qwen_lora2.json",
    "mistral_base": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/mistral/judgments_mistral-small-3.1-24b-instruct-2503_dialogues_ministral_base.json",
    "mistral_lora1": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/mistral/judgments_mistral-small-3.1-24b-instruct-2503_dialogues_ministral_lora1.json",
    "mistral_lora2": "C:/Users/Tim/Projects/MA/data/llmasjudge_evals_pub/mistral/judgments_mistral-small-3.1-24b-instruct-2503_dialogues_ministral_lora2.json"
}

for variant_name, filepath in eval_files.items():
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Konnte Datei nicht laden: {filepath} -> {e}")
        continue
        
    rows = []
    
  # extract valid evals
    clean_dialogues = extract_valid_dialogues(data)
    
    for diag in clean_dialogues:
        global_id = diag.get("global_id")
        conclusion = diag.get("final_conclusion", {})
        
        pred_a = conclusion.get("person_A", {}).get("type (best guess)")
        pred_b = conclusion.get("person_B", {}).get("type (best guess)")
        
        rows.append({
            "global_id": global_id,
            f"pred_A_{variant_name}": pred_a,
            f"pred_B_{variant_name}": pred_b
        })
        
    df_temp = pd.DataFrame(rows)
    
    # merge table with global_id
    df_master_mistral31_24B = pd.merge(df_master_mistral31_24B, df_temp, on="global_id", how="left")

display(df_master_mistral31_24B.head())

,global_id,Szenario,Run,true_MBTI_A,true_MBTI_B,pred_A_qwen_base,pred_B_qwen_base,pred_A_qwen_lora1,pred_B_qwen_lora1,pred_A_qwen_lora2,pred_B_qwen_lora2,pred_A_mistral_base,pred_B_mistral_base,pred_A_mistral_lora1,pred_B_mistral_lora1,pred_A_mistral_lora2,pred_B_mistral_lora2
0,1,Work Place - Low Urgency,1,INTJ,ENFP,ENTJ,ENFJ,ISTP,ESFJ,ESTJ,ESFJ,INTJ,ENFP,ENTJ,ESFJ,ENTJ,INFJ
1,2,Work Place - Low Urgency,2,INTJ,ENFP,ISTJ,ISFJ,ENFP,ISFJ,None,None,ENTJ,ENFJ,ENTP,INFP,ISTJ,ENFJ
2,3,Work Place - Low Urgency,3,INTJ,ENFP,ENTJ,ISFJ,ENTJ,ISFJ,None,None,ISTJ,ENFP,ISTJ,ISFP,ESFJ,INFP
3,4,Work Place - Low Urgency,4,INTJ,ENFP,ISTJ,ESFJ,ENTP,ENFP,INFP,ENFP,ENTJ,ENFP,ESTJ,ISFP,ENTJ,ISFP
4,5,Work Place - Low Urgency,5,INTJ,ENFP,None,None,ENFJ,ESFJ,ESFJ,INFJ,ISTJ,ENFP,ISTJ,INFP,ENFP,ISTJ


In [ ]:
# cleaning function
def clean_mbti(val):
    if pd.isna(val) or not isinstance(val, str):
        return None
    clean_val = val.strip().upper()
    match = re.search(r'[IE][SN][TF][JP]', clean_val)
    if match:
        return match.group(0)
    return None

variant_names = [
    "mistral_base", "mistral_lora1", "mistral_lora2", 
    "qwen_base", "qwen_lora1", "qwen_lora2"
]

dim_names = ['E/I', 'S/N', 'T/F', 'J/P']
results = []

for var in variant_names:
    true_labels = df_master_mistral31_24B['true_MBTI_A'].tolist() + df_master_mistral31_24B['true_MBTI_B'].tolist()
    pred_col_A = f"pred_A_{var}"
    pred_col_B = f"pred_B_{var}"
    
    if pred_col_A not in df_master_mistral31_24B.columns or pred_col_B not in df_master_mistral31_24B.columns:
        continue
        
    pred_labels = df_master_mistral31_24B[pred_col_A].apply(clean_mbti).tolist() + \
                  df_master_mistral31_24B[pred_col_B].apply(clean_mbti).tolist()
    
    # filter valid pairs
    valid_true = []
    valid_pred = []
    for t, p in zip(true_labels, pred_labels):
        if p is not None and t is not None and len(t) == 4 and len(p) == 4:
            valid_true.append(t)
            valid_pred.append(p)
            
    if len(valid_true) == 0:
        continue
        
    # base data for this row
    row_data = {
        "Model Variant": var.replace("_", " ").title(),
        "N": len(valid_true)
    }
    
    # evals per dimension
    for i, dim in enumerate(dim_names):

        t_dim = [t[i] for t in valid_true]
        p_dim = [p[i] for p in valid_pred]
        
        acc = accuracy_score(t_dim, p_dim)

        f1 = f1_score(t_dim, p_dim, average='macro')
        
        row_data[f"{dim} Acc"] = round(acc, 3)
        row_data[f"{dim} F1"] = round(f1, 3)
        
    results.append(row_data)


df_results_mistral31_24B = pd.DataFrame(results)
display(df_results_mistral31_24B)

,Model Variant,N,E/I Acc,E/I F1,S/N Acc,S/N F1,T/F Acc,T/F F1,J/P Acc,J/P F1
0,Mistral Base,427,0.569,0.569,0.621,0.621,0.768,0.767,0.735,0.730
1,Mistral Lora1,433,0.471,0.471,0.478,0.475,0.589,0.589,0.577,0.573
2,Mistral Lora2,440,0.482,0.478,0.482,0.471,0.641,0.641,0.636,0.634
3,Qwen Base,399,0.559,0.558,0.574,0.570,0.739,0.737,0.667,0.647
4,Qwen Lora1,443,0.470,0.469,0.540,0.528,0.693,0.690,0.612,0.605
5,Qwen Lora2,430,0.486,0.481,0.535,0.528,0.665,0.661,0.581,0.578


In [ ]:
from collections import Counter
import numpy as np
import pandas as pd
import re
from sklearn.metrics import accuracy_score, f1_score

def clean_mbti(val):
    if pd.isna(val) or not isinstance(val, str): return None
    match = re.search(r'[IE][SN][TF][JP]', val.strip().upper())
    return match.group(0) if match else None

# master tables of the judges
judges_dfs = [df_master_mistral31_24B, df_master_qwen3_30B, df_master_qwen3_32B]

# ground truth (identical in every master table)
base_gt_df = df_master_qwen3_32B 

variant_names = ["mistral_base", "mistral_lora1", "mistral_lora2", "qwen_base", "qwen_lora1", "qwen_lora2"]
dim_names = ['E/I', 'S/N', 'T/F', 'J/P']

consensus_results = []

for var in variant_names:
    true_labels_A = base_gt_df['true_MBTI_A'].tolist() if 'true_MBTI_A' in base_gt_df.columns else base_gt_df['MBTI_A'].tolist()
    true_labels_B = base_gt_df['true_MBTI_B'].tolist() if 'true_MBTI_B' in base_gt_df.columns else base_gt_df['MBTI_B'].tolist()
    
    col_A = f"pred_A_{var}"
    col_B = f"pred_B_{var}"
    
    row_data = {"Model Variant": var.replace("_", " ").title()}
    dialogue_count = 0
    
    dim_true_all = {dim: [] for dim in dim_names}
    dim_pred_consensus = {dim: [] for dim in dim_names}
    
    for row_idx in range(len(base_gt_df)):
        t_A = true_labels_A[row_idx]
        t_B = true_labels_B[row_idx]
        
        cleaned_t_A = clean_mbti(t_A)
        cleaned_t_B = clean_mbti(t_B)
        
        if not cleaned_t_A or not cleaned_t_B:
            continue
            
        preds_A_list = []
        preds_B_list = []
        for df_j in judges_dfs:
            if col_A in df_j.columns:
                pA = clean_mbti(df_j.iloc[row_idx][col_A])
                if pA and len(pA) == 4: preds_A_list.append(pA)
            if col_B in df_j.columns:
                pB = clean_mbti(df_j.iloc[row_idx][col_B])
                if pB and len(pB) == 4: preds_B_list.append(pB)
                
        # if at least one judge has a valid example we use that one
        if len(preds_A_list) > 0 and len(preds_B_list) > 0:
            dialogue_count += 1
            
            for i, dim in enumerate(dim_names):
                votes_A = [p[i] for p in preds_A_list]
                consensus_A = Counter(votes_A).most_common(1)[0][0]
                
                dim_true_all[dim].append(cleaned_t_A[i])
                dim_pred_consensus[dim].append(consensus_A)
                
                votes_B = [p[i] for p in preds_B_list]
                consensus_B = Counter(votes_B).most_common(1)[0][0]
                
                dim_true_all[dim].append(cleaned_t_B[i])
                dim_pred_consensus[dim].append(consensus_B)

    row_data["N"] = dialogue_count
    
    for dim in dim_names:
        t_list = dim_true_all[dim]
        p_list = dim_pred_consensus[dim]
        if len(t_list) > 0:
            acc = accuracy_score(t_list, p_list)
            f1 = f1_score(t_list, p_list, average='macro')
            row_data[f"{dim} Acc"] = round(acc, 3)
            row_data[f"{dim} F1"] = round(f1, 3)
        else:
            row_data[f"{dim} Acc"] = 0.0
            row_data[f"{dim} F1"] = 0.0
            
    consensus_results.append(row_data)

df_consensus = pd.DataFrame(consensus_results)
display(df_consensus)

,Model Variant,N,E/I Acc,E/I F1,S/N Acc,S/N F1,T/F Acc,T/F F1,J/P Acc,J/P F1
0,Mistral Base,225,0.596,0.595,0.620,0.607,0.820,0.820,0.760,0.758
1,Mistral Lora1,225,0.462,0.462,0.500,0.486,0.738,0.738,0.633,0.633
2,Mistral Lora2,225,0.449,0.446,0.524,0.517,0.729,0.729,0.687,0.687
3,Qwen Base,225,0.533,0.530,0.604,0.596,0.791,0.791,0.729,0.721
4,Qwen Lora1,225,0.458,0.455,0.529,0.525,0.778,0.777,0.673,0.672
5,Qwen Lora2,225,0.429,0.425,0.549,0.543,0.758,0.758,0.669,0.668


In [ ]:
import numpy as np
import pandas as pd
import krippendorff
import re

# cleaning funciton
def clean_mbti(val):
    if pd.isna(val) or not isinstance(val, str):
        return None
    clean_val = val.strip().upper()
    match = re.search(r'[IE][SN][TF][JP]', clean_val)
    return match.group(0) if match else None

# parameter setup
judges_dfs = [df_master_mistral31_24B, df_master_qwen3_30B, df_master_qwen3_32B]

variant_names = [
    "mistral_base", "mistral_lora1", "mistral_lora2", 
    "qwen_base", "qwen_lora1", "qwen_lora2"
]

dim_mappings = {
    0: {'I': 0, 'E': 1},
    1: {'N': 0, 'S': 1},
    2: {'F': 0, 'T': 1},
    3: {'P': 0, 'J': 1}
}
dim_names = ['E/I', 'S/N', 'T/F', 'J/P']

print("Inter-Rater Reliability (3 LLM Judges)")
print("-" * 40)
print(f"{'Dimension':<10} {'Krippendorff´s α':>20}")
print("-" * 40)

# alpha per dimension
for dim_idx, dim_name in enumerate(dim_names):
    mapping = dim_mappings[dim_idx]
    
    # initialise matrix
    rater_matrix = [[], [], []]
    
    for variant in variant_names:
        col_A = f"pred_A_{variant}"
        col_B = f"pred_B_{variant}"
        
        for row_idx in range(len(judges_dfs[0])):
            for col in [col_A, col_B]:
                
                # get labels of all judges
                for rater_idx, df_judge in enumerate(judges_dfs):
                    
                    if col not in df_judge.columns:
                        rater_matrix[rater_idx].append(np.nan)
                        continue
                        
                    val = df_judge.iloc[row_idx][col]
                    cleaned_val = clean_mbti(val)
                    
                    if cleaned_val and len(cleaned_val) == 4:
                        char = cleaned_val[dim_idx]
                        num_val = mapping.get(char, np.nan)
                    else:
                        num_val = np.nan
                        
                    rater_matrix[rater_idx].append(num_val)
                    
    rater_matrix = np.array(rater_matrix)
    
    try:
        alpha = krippendorff.alpha(reliability_data=rater_matrix, level_of_measurement='nominal')
        print(f"{dim_name:<10} {alpha:>20.4f}")
    except ValueError:
        print(f"{dim_name:<10} {'no variance/error':>20}")

print("-" * 40)

Inter-Rater Reliability (3 LLM Judges)
----------------------------------------
Dimension      Krippendorff´s α
----------------------------------------
E/I                      0.4516
S/N                      0.0720
T/F                      0.5524
J/P                      0.4929
----------------------------------------


In [ ]:
import numpy as np
import pandas as pd
import krippendorff
import re

def clean_mbti(val):
    if pd.isna(val) or not isinstance(val, str):
        return None
    clean_val = val.strip().upper()
    match = re.search(r'[IE][SN][TF][JP]', clean_val)
    return match.group(0) if match else None

judges_dfs = [df_master_mistral31_24B, df_master_qwen3_30B, df_master_qwen3_32B]

variant_names = [
    "mistral_base", "mistral_lora1", "mistral_lora2", 
    "qwen_base", "qwen_lora1", "qwen_lora2"
]

dim_mappings = {
    0: {'I': 0, 'E': 1},
    1: {'N': 0, 'S': 1},
    2: {'F': 0, 'T': 1},
    3: {'P': 0, 'J': 1}
}
dim_names = ['E/I', 'S/N', 'T/F', 'J/P']

alpha_results = []

judges_indexed = [
    df.set_index("global_id")
    for df in judges_dfs
]

for dim_idx, dim_name in enumerate(dim_names):

    mapping = dim_mappings[dim_idx]
    row_data = {"Dimension": dim_name}

    for variant in variant_names:

        col_A = f"pred_A_{variant}"
        col_B = f"pred_B_{variant}"

        rater_matrix = [[], [], []]

        # all dialogue ids
        all_ids = sorted(
            set().union(
                *(df.index for df in judges_indexed)
            )
        )

        for global_id in all_ids:

            # Person A und Person B separate
            for col in [col_A, col_B]:

                for rater_idx, df_judge in enumerate(judges_indexed):

                    # if judge hasn't labelled this dialogue
                    if global_id not in df_judge.index:
                        rater_matrix[rater_idx].append(np.nan)
                        continue

                    # col doesn't exist
                    if col not in df_judge.columns:
                        rater_matrix[rater_idx].append(np.nan)
                        continue

                    val = df_judge.loc[global_id, col]

                    cleaned_val = clean_mbti(val)

                    if cleaned_val and len(cleaned_val) == 4:
                        char = cleaned_val[dim_idx]
                        num_val = mapping.get(char, np.nan)
                    else:
                        num_val = np.nan

                    rater_matrix[rater_idx].append(num_val)

        rater_matrix = np.array(rater_matrix)

        try:
            alpha = krippendorff.alpha(
                reliability_data=rater_matrix,
                level_of_measurement="nominal"
            )

            row_data[variant] = round(alpha, 3)

        except ValueError:
            row_data[variant] = np.nan

    alpha_results.append(row_data)

df_alpha = pd.DataFrame(alpha_results)

display(df_alpha)

,Dimension,mistral_base,mistral_lora1,mistral_lora2,qwen_base,qwen_lora1,qwen_lora2
0,E/I,0.435,0.452,0.425,0.428,0.555,0.406
1,S/N,0.164,0.086,0.080,0.024,0.045,0.027
2,T/F,0.621,0.453,0.487,0.691,0.540,0.523
3,J/P,0.654,0.382,0.483,0.583,0.457,0.393


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
import pandas as pd
import re

def clean_mbti(val):
    if pd.isna(val) or not isinstance(val, str): return None
    match = re.search(r'[IE][SN][TF][JP]', val.strip().upper())
    return match.group(0) if match else None

judges_dfs = [df_master_mistral31_24B, df_master_qwen3_30B, df_master_qwen3_32B]

# model comparisons
comparisons = [
    ("mistral_base", "mistral_lora1"),
    ("mistral_base", "mistral_lora2"),
    ("qwen_base", "qwen_lora1"),
    ("qwen_base", "qwen_lora2")
]

dim_names = ['E/I', 'S/N', 'T/F', 'J/P']

for base_model, tuned_model in comparisons:
    print(f"=== McNemar: {base_model} vs {tuned_model} ===")
    print(f"{'Dim':<6} {'b (Tuned \u2713)':>14} {'c (Base \u2713)':>14} {'p-value':>10} {'n_needed':>10}")
    print("-" * 60)
    
    for dim_idx, dim in enumerate(dim_names):
        all_tuned_correct = []
        all_base_correct = []
        
        # pooling data of the judges
        for df in judges_dfs:
            # Put persons A and B together
            true_labels = df['true_MBTI_A'].tolist() + df['true_MBTI_B'].tolist()
            pred_base = df[f'pred_A_{base_model}'].apply(clean_mbti).tolist() + df[f'pred_B_{base_model}'].apply(clean_mbti).tolist()
            pred_tuned = df[f'pred_A_{tuned_model}'].apply(clean_mbti).tolist() + df[f'pred_B_{tuned_model}'].apply(clean_mbti).tolist()
            
            # analysis per dimension
            for t, p_base, p_tuned in zip(true_labels, pred_base, pred_tuned):
                if t and p_base and p_tuned and len(t)==4 and len(p_base)==4 and len(p_tuned)==4:
                    all_base_correct.append(1 if p_base[dim_idx] == t[dim_idx] else 0)
                    all_tuned_correct.append(1 if p_tuned[dim_idx] == t[dim_idx] else 0)
                    
        a_correct = np.array(all_tuned_correct)
        b_correct = np.array(all_base_correct)
        
        # discordant pairs
        b = ((a_correct == 1) & (b_correct == 0)).sum() # Tuned hat recht, Base irrt
        c = ((a_correct == 0) & (b_correct == 1)).sum() # Tuned irrt, Base hat recht
        
        if b + c == 0:
            print(f"{dim:<6} {'no discordant pairs':>45}")
            continue
            
        # McNemar
        table = np.array([
            [((a_correct == 1) & (b_correct == 1)).sum(), b],
            [c, ((a_correct == 0) & (b_correct == 0)).sum()]
        ])
        result = mcnemar(table, exact=True)
        
        # Power analysis
        p_b = b / (b + c)
        p_c = c / (b + c)
        if p_b != p_c:
            n_needed = int(np.ceil((1.96 + 0.842)**2 * (p_b + p_c) / (p_b - p_c)**2))
        else:
            n_needed = np.inf
            
        print(f"{dim:<6} {b:>14} {c:>14} {result.pvalue:>10.4f} {n_needed:>10}")
    print("\n")
    

=== McNemar: mistral_base vs mistral_lora1 ===
Dim       b (Tuned ✓)     c (Base ✓)    p-value   n_needed
------------------------------------------------------------
E/I               147            304     0.0000         65
S/N               170            286     0.0000        122
T/F                98            233     0.0000         48
J/P               109            288     0.0000         39


=== McNemar: mistral_base vs mistral_lora2 ===
Dim       b (Tuned ✓)     c (Base ✓)    p-value   n_needed
------------------------------------------------------------
E/I               155            316     0.0000         68
S/N               195            296     0.0000        186
T/F               100            207     0.0000         65
J/P               126            251     0.0000         72


=== McNemar: qwen_base vs qwen_lora1 ===
Dim       b (Tuned ✓)     c (Base ✓)    p-value   n_needed
------------------------------------------------------------
E/I               149        